# 06b — Avaliação do RF binário no teste (variante COM os atributos de AUC alto)

> **Variante COM os atributos de AUC alto.** Avalia o modelo final do notebook 05b na view de teste do notebook 03b (mesmas linhas de teste do notebook 06, com as 3 colunas a mais). Para comparar com a versão sem esses atributos, use os dois `metricas_teste.json`; **a escolha da versão deve seguir a regra definida antes, pela validação**, e não pelo teste. Todos os objetos criados aqui têm o sufixo `_com_auc`: **nada** da versão sem esses atributos é alterado.

Este notebook avalia o **modelo final** (o do refit, notebook 05b) no **teste**. **Use-o uma vez.** Se você mexer no modelo, nos atributos ou na configuração depois de ver estes números, o teste deixa de ser uma avaliação independente. Por isso o notebook **para** se já houver resultado gravado para essa execução (`PERMITIR_REPETIR = True` desativa a trava).

## O que ele faz
1. Carrega o modelo, o pré-processamento e os limiares da validação gravados pelo notebook 05b (a pasta `rf_bin_<data>` mais recente, ou `PASTA_RUN`).
2. Lê `prep_vw_rf_bin_teste_sem_sobrep_com_auc` em **lotes de 50 mil linhas** por faixas de `id_linha` (o teste tem milhões de linhas; a leitura é econômica em memória), prevê cada lote e **grava o resultado do lote em disco**. Se a execução for interrompida, é só rodar de novo: os lotes prontos não são refeitos.
3. Calcula as métricas **no teste como ele é** (sem balancear, sem pesos).
4. Mede o custo de inferência (latência, vazão) e registra tudo.

## Como ler
- **O teste não é balanceado** e não tem as linhas que se repetem do treino (opção A). Tem ~0,8% de BENIGN.
- **Dois limiares:** **0,5** (padrão) e os **limiares da validação** para falso alarme de 1% e de 0,1%, escolhidos **antes** de ver o teste. O falso alarme que realmente sai no teste pode diferir um pouco, e isso é um resultado. Os limiares vieram do modelo da fase 1; o modelo final foi refeito com mais dados, então a calibração pode mudar levemente.
- Com ~99% de ataques, a **precisão, o F1 e o PR-AUC dos ataques** ficam perto de 1 mesmo para um modelo fraco. Olhe o **MCC**, a **taxa de falso alarme** (BENIGN chamado de ataque) e as métricas do **BENIGN** (`precisao_benign`, `f1_benign`, `pr_auc_benign`; a linha de base do PR-AUC do BENIGN é a proporção de BENIGN no teste).
- O ponto "limiar escolhido no próprio teste" (recall com falso alarme de 1% lido na curva ROC do teste) é **só informativo**.

## Limitações para registrar no TCC
- Dentro do próprio teste há muitas linhas repetidas (a EDA indicou ~89% de redundância): poucos padrões repetidos pesam muito na nota.
- No binário **não** se calcula recall por tipo de ataque (decisão sua): uma queda num tipo específico fica escondida na média.
- O custo de inferência aqui é do **modelo** (com o pré-processamento); a extração das características do tráfego não entra.

## Antes de rodar
O notebook 05b precisa ter terminado. Coloque este arquivo na **mesma pasta** dos outros (onde está o `.env`). Se o computador tiver pouca memória livre, **feche os outros notebooks** (Kernel → Shutdown) e programas pesados: cada kernel aberto guarda a sua memória.

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
def _div(a, b): return float(a) / float(b) if b else float("nan")

def metricas(y, p, limiar=0.5, w=None):
    # Todas as métricas de uma vez. Com `w` (pesos), as contagens e as métricas que dependem da proporção ficam ponderadas.
    # O ROC-AUC é calculado sem pesos: ele só compara ataques com BENIGN e não depende da proporção.
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float); pred = (p >= limiar).astype(int)
    w_ = np.ones(len(y)) if w is None else np.asarray(w, dtype=float)
    tp = float(w_[(y == 1) & (pred == 1)].sum()); tn = float(w_[(y == 0) & (pred == 0)].sum())
    fp = float(w_[(y == 0) & (pred == 1)].sum()); fn = float(w_[(y == 1) & (pred == 0)].sum())
    rec, esp, prec = _div(tp, tp + fn), _div(tn, tn + fp), _div(tp, tp + fp)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    f1 = _div(2 * prec * rec, prec + rec) if (prec == prec and rec == rec) else float("nan")
    prec_b = _div(tn, tn + fn); f1_b = _div(2 * prec_b * esp, prec_b + esp) if (prec_b == prec_b and esp == esp) else float("nan")    # classe BENIGN como "positiva"
    return {"acuracia": _div(tp + tn, tp + tn + fp + fn), "acuracia_balanceada": (rec + esp) / 2, "precisao": prec, "recall": rec, "especificidade": esp,
            "taxa_falso_alarme": 1 - esp, "taxa_perda": 1 - rec, "f1": f1, "mcc": _div(tp * tn - fp * fn, den),
            "precisao_benign": prec_b, "f1_benign": f1_b, "f1_macro": (f1 + f1_b) / 2, "pr_auc_benign": float(average_precision_score(1 - y, 1 - p, sample_weight=w_)),
            "roc_auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p, sample_weight=w_)), "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def recall_em_fpr(y, p, alvo):
    # Ponto de operação: o MAIOR recall com taxa de falso alarme <= alvo (limiar escolhido nestes dados).
    fpr, tpr, thr = roc_curve(np.asarray(y).astype(int), np.asarray(p, dtype=float))
    i = max(int(np.searchsorted(fpr, alvo, side="right")) - 1, 0)
    lim = float(thr[i]) if np.isfinite(thr[i]) else 1.0 + 1e-9
    return {"alvo_fpr": alvo, "limiar": lim, "recall": float(tpr[i]), "fpr": float(fpr[i])}

class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# autoteste das ferramentas (não toca no servidor)
_y = np.array([0, 0, 0, 0, 1, 1, 1, 1]); _p = np.array([.1, .2, .3, .6, .4, .7, .8, .9]); _m = metricas(_y, _p, 0.5)
assert (_m["tp"], _m["fp"], _m["tn"], _m["fn"]) == (3, 1, 3, 1) and abs(_m["mcc"] - 0.5) < 1e-12 and abs(_m["acuracia"] - 0.75) < 1e-12, _m
_o = recall_em_fpr(_y, _p, 0.25); assert abs(_o["recall"] - 1.0) < 1e-12 and _o["fpr"] <= 0.25 and abs(_o["limiar"] - 0.4) < 1e-12, _o
_m2 = metricas(_y, _p, 0.5); assert abs(_m2["precisao_benign"] - 0.75) < 1e-12 and abs(_m2["f1_benign"] - 0.75) < 1e-12 and abs(_m2["f1_macro"] - 0.75) < 1e-12, _m2
_o0 = recall_em_fpr(_y, _p, 0.0); assert abs(_o0["recall"] - 0.75) < 1e-12 and _o0["fpr"] == 0.0, _o0
_d = distribuir(100, {"a": 1000, "b": 1000, "c": 1000, "d": 10}, 0.0, {"a": 1000, "b": 1000, "c": 1000, "d": 10}); assert _d["d"] == 10 and sum(_d.values()) == 100, _d
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


### Leitura econômica em memória
O leitor `consultar` baixa o resultado como JSON e cria um objeto Python para cada número: **um lote de 250 mil linhas chega a gastar ~750 MiB**. Aqui o teste é lido por `ler_csv`, que recebe CSV e entrega direto ao pandas (**~200 MiB** para as mesmas 250 mil linhas, e ~40 MiB para 50 mil), com os mesmos números. Se o seu computador tiver pouca memória livre, o JSON pode fazer o kernel do Jupyter **morrer** sem mensagem de erro.

A leitura também é **leve para o servidor** (`max_threads=1`, blocos pequenos). Se mesmo assim o ClickHouse responder `Code 241` (servidor sem memória, comum com 3 GiB e merges em andamento), `ler_csv_com_espera` **espera e tenta de novo** (30 s, 60 s, 2 min, 4 min e 8 min; sai antes se a memória do servidor cair abaixo de 2 GiB; ajustável com `ESPERAS_MEMORIA`). Se esgotar, é só **rodar a célula de novo mais tarde**: os lotes prontos são reaproveitados.

### Métricas leves em memória
Com ~7,2 milhões de previsões, as funções do sklearn ordenam cópias dos dados várias vezes (**~650 MiB** a mais para 3 limiares, no meu teste) e podem dar `MemoryError`. As métricas aqui saem de **contagens de ataques e BENIGN por valor de escore** (um RF de 100 árvores tem só ~100 valores): o resultado é **idêntico** ao do sklearn (diferença máxima de 2×10⁻¹⁶ nos meus testes) e gasta quase nenhuma memória.

In [6]:
_trap = getattr(np, "trapezoid", None) or getattr(np, "trapz")

def contagens_por_escore(y, p, bloco=1_000_000):
    # Conta ataques (y = 1) e BENIGN (y = 0) por VALOR DISTINTO do escore. É tudo o que as métricas precisam e ocupa memória mínima
    # (as funções do sklearn ordenam cópias de 7 milhões de elementos várias vezes). Os escores são comparados em float32, como foram gravados.
    y = np.asarray(y); p = np.asarray(p, dtype=np.float32)
    vals = np.unique(p)
    if len(vals) > 200_000:                                            # escores quase contínuos: caminho de uma passada
        vals, inv = np.unique(p, return_inverse=True)
        n_tot = np.bincount(inv, minlength=len(vals)); n_pos = np.bincount(inv, weights=y, minlength=len(vals))
    else:
        n_tot = np.zeros(len(vals), np.int64); n_pos = np.zeros(len(vals), np.float64)
        for i in range(0, len(p), bloco):
            idx = np.searchsorted(vals, p[i:i + bloco]); n_tot += np.bincount(idx, minlength=len(vals)); n_pos += np.bincount(idx, weights=y[i:i + bloco], minlength=len(vals))
    n_pos = np.rint(n_pos).astype(np.int64); n_tot = n_tot.astype(np.int64)
    return vals, n_pos, n_tot - n_pos

def _curva(tp_c, fp_c, P, N):
    tpr = np.concatenate(([0.0], tp_c / P)); fpr = np.concatenate(([0.0], fp_c / N)); rec = tp_c / P; prec = tp_c / (tp_c + fp_c)
    return fpr, tpr, float(_trap(tpr, fpr)), float(np.sum(np.diff(np.concatenate(([0.0], rec))) * prec))

def metricas_hist(vals, n_pos, n_neg, limiar=0.5):
    # Mesmas métricas de `metricas` (sem pesos), a partir das contagens por escore. Previsão = escore >= limiar (ambos em float32).
    m = vals >= np.float32(limiar)
    tp, fp, fn, tn = float(n_pos[m].sum()), float(n_neg[m].sum()), float(n_pos[~m].sum()), float(n_neg[~m].sum())
    P, N = tp + fn, tn + fp
    rec, esp, prec = _div(tp, P), _div(tn, N), _div(tp, tp + fp); prec_b = _div(tn, tn + fn)
    f1 = _div(2 * prec * rec, prec + rec) if (prec == prec and rec == rec) else float("nan")
    f1_b = _div(2 * prec_b * esp, prec_b + esp) if (prec_b == prec_b and esp == esp) else float("nan")
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)); d = slice(None, None, -1)
    _, _, roc_auc, pr_a = _curva(np.cumsum(n_pos[d]), np.cumsum(n_neg[d]), P, N)       # ataque como positivo (escores em ordem decrescente)
    _, _, _, pr_b = _curva(np.cumsum(n_neg), np.cumsum(n_pos), N, P)                    # BENIGN como positivo (escore 1 - p: ordem crescente de p)
    return {"acuracia": _div(tp + tn, tp + tn + fp + fn), "acuracia_balanceada": (rec + esp) / 2, "precisao": prec, "recall": rec, "especificidade": esp,
            "taxa_falso_alarme": 1 - esp, "taxa_perda": 1 - rec, "f1": f1, "mcc": _div(tp * tn - fp * fn, den), "precisao_benign": prec_b, "f1_benign": f1_b,
            "f1_macro": (f1 + f1_b) / 2, "pr_auc_benign": pr_b, "roc_auc": roc_auc, "pr_auc": pr_a, "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def recall_em_fpr_hist(vals, n_pos, n_neg, alvo):
    # O MAIOR recall com taxa de falso alarme <= alvo, lido na curva ROC exata (todos os pontos distintos).
    d = slice(None, None, -1); P, N = n_pos.sum(), n_neg.sum()
    tpr = np.concatenate(([0.0], np.cumsum(n_pos[d]) / P)); fpr = np.concatenate(([0.0], np.cumsum(n_neg[d]) / N)); thr = np.concatenate(([np.inf], vals[d].astype(float)))
    i = max(int(np.searchsorted(fpr, alvo, side="right")) - 1, 0); lim = float(thr[i]) if np.isfinite(thr[i]) else 1.0 + 1e-9
    return {"alvo_fpr": alvo, "limiar": lim, "recall": float(tpr[i]), "fpr": float(fpr[i])}

In [7]:
import gc

def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

class ErroMemoriaServidor(Exception):
    pass


CH_SETTINGS_LEITURA = globals().get("CH_SETTINGS_LEITURA", {"max_threads": 1, "max_block_size": 8192})      # leitura leve para o servidor (menos colunas e blocos abertos ao mesmo tempo)
ESPERAS_MEMORIA = globals().get("ESPERAS_MEMORIA", (30, 60, 120, 240, 480))                            # segundos de espera após "sem memória" (Code 241) no servidor
LIMITE_MEM_SERVIDOR_GIB = globals().get("LIMITE_MEM_SERVIDOR_GIB", 2.0)                                        # volta a tentar antes do fim da espera se a memória do servidor cair abaixo disto


def ler_csv(query, colunas):
    # Leitura em FLUXO: o servidor devolve CSV e o pandas lê direto da conexão, sem guardar o texto inteiro nem criar milhões de objetos Python
    # (o caminho JSON do consultar() gasta várias vezes mais memória). Mesmas regras de repetição do _http: só SELECT, com novas tentativas.
    global _conn
    if not re.match(r"\s*SELECT\b", query, re.I): raise PermissionError("ler_csv só executa SELECT")
    caminho = "/?" + urllib.parse.urlencode({"database": CH_DATABASE, "default_format": "CSV", "wait_end_of_query": 1, **CH_SETTINGS_LEITURA})
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    ultimo = None
    for espera in CONEXAO_ESPERAS:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s..."); time.sleep(espera)
        corpo = b""
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            if resp.status == 200:
                try:
                    return pd.read_csv(resp, header=None, names=colunas, dtype=np.float64, na_values=["\\N"])
                except pd.errors.EmptyDataError:
                    return pd.DataFrame({c: pd.Series(dtype=np.float64) for c in colunas})
            corpo = resp.read()
        except (OSError, http.client.HTTPException, pd.errors.ParserError, ValueError) as e:
            ultimo = e; _fechar_conexao(); continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} corpo={corpo[:300].decode('utf-8', 'replace')!r}"); _fechar_conexao(); continue
        texto = corpo.decode("utf-8", "replace")
        if "MEMORY_LIMIT_EXCEEDED" in texto or texto.startswith("Code: 241"):
            raise ErroMemoriaServidor(texto[:600])
        raise Exception(texto[:2000])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(CONEXAO_ESPERAS)} tentativas: {ultimo!r}")


def _mem_servidor_gib():
    try:
        return float(consultar_num("SELECT toFloat64(value) AS v FROM system.asynchronous_metrics WHERE metric = 'MemoryResident'")["v"][0]) / 2 ** 30
    except Exception:
        return None


def ler_csv_com_espera(query, colunas):
    # Se o servidor ficar sem memória (Code 241, comum num servidor de 3 GiB com merges em andamento), espera e tenta de novo; o erro é transitório.
    for k, espera in enumerate(list(ESPERAS_MEMORIA) + [None]):
        try:
            return ler_csv(query, colunas)
        except ErroMemoriaServidor as e:
            if espera is None:
                raise ErroMemoriaServidor("O servidor continua sem memória depois de todas as esperas. Rode a célula de novo mais tarde: os lotes já prontos são reaproveitados. Último erro: " + str(e)[:300]) from None
            m = _mem_servidor_gib()
            print(f"  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: {'?' if m is None else f'{m:.2f} GiB'}); esperando até {espera}s e tentando de novo ({k + 1}/{len(ESPERAS_MEMORIA)})...", flush=True)
            t0 = time.time()
            while time.time() - t0 < espera:
                time.sleep(min(15, espera))
                m = _mem_servidor_gib()
                if m is not None and m < LIMITE_MEM_SERVIDOR_GIB: break

## 1. Carregar o modelo e conferir

Confere que a **ordem das colunas** do modelo é a da view de teste (se não for, as previsões ficariam trocadas sem aviso) e que a trava de uso único não foi acionada.

In [8]:
# ---------- 1.1 Parâmetros e carga ----------
PASTA_BASE = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "rf_binario_com_auc"
PASTA_RUN = globals().get("PASTA_RUN", None)
if PASTA_RUN is None:
    runs = sorted(p for p in PASTA_BASE.glob("rf_bin_*") if (p / "modelo_final_refit.joblib").exists())
    assert runs, "nenhuma pasta rf_bin_* com modelo_final_refit.joblib em " + str(PASTA_BASE) + ": rode antes o notebook 05b"
    PASTA_RUN = runs[-1]
PASTA_RUN = Path(PASTA_RUN)
VIEW_TESTE = "prep_vw_rf_bin_teste_sem_sobrep_com_auc"
LOTE_LINHAS = globals().get("LOTE_LINHAS", 50_000)          # ~40 MiB de memória por lote; a tabela é ordenada por id_linha, então muitos lotes pequenos são baratos para o servidor
PERMITIR_REPETIR = globals().get("PERMITIR_REPETIR", False)
N_LATENCIA = globals().get("N_LATENCIA", 1000)
SEMENTE = globals().get("SEMENTE", 42)
LIMIAR = 0.5

RESULTADO = PASTA_RUN / "metricas_teste.json"
if RESULTADO.exists() and not PERMITIR_REPETIR:
    raise RuntimeError(f"O teste já foi avaliado para esta execução ({RESULTADO.name}). Repetir o teste depois de ajustar o modelo o invalida como avaliação independente. "
                       "Para repetir de propósito (por exemplo, para refazer os gráficos), defina PERMITIR_REPETIR = True antes desta célula.")
assert existe(VIEW_TESTE), f"a view {VIEW_TESTE} não existe: rode o notebook 03b"

modelo = joblib.load(PASTA_RUN / "modelo_final_refit.joblib")
prep_cfg = json.loads((PASTA_RUN / "preprocessamento.json").read_text(encoding="utf-8"))
config = json.loads((PASTA_RUN / "config_treino.json").read_text(encoding="utf-8"))
lim_val = json.loads((PASTA_RUN / "limiares_validacao.json").read_text(encoding="utf-8"))
prep = PreprocessadorRF(prep_cfg["features"], prep_cfg["medianas_treino"], prep_cfg["colunas_retiradas_ablacao"], prep_cfg["versao_views_rf"])
CM = prep.colunas_modelo
assert CM == prep_cfg["colunas_modelo"] == list(modelo.colunas_modelo_), "a ordem das colunas do modelo difere da gravada"
cols_view = colunas_view(VIEW_TESTE)
assert cols_view[:2] == ["id_linha", "y"] and cols_view[2:] == CM, "as colunas da view de teste não coincidem com as do modelo (nome ou ordem)"
assert all(getattr(modelo, k) == v for k, v in config["hiperparametros"].items()), "o modelo carregado não tem os hiperparâmetros gravados"
N_TESTE = n_linhas(VIEW_TESTE)
print(f"Execução avaliada: {PASTA_RUN.name} | treinada em {config['data']} | {len(CM)} atributos | views do notebook 03b versão {config['versao_views_notebook03']}")
print(f"Teste (sem sobreposição com o treino): {N_TESTE:,} linhas")
print("Limiares gravados na validação:", [(p['alvo_fpr'], round(p['limiar'], 4)) for p in lim_val["pontos_de_operacao_validacao"]])

Execução avaliada: rf_bin_20261024_002027 | treinada em 2026-10-24T00:24:01 | 46 atributos | views do notebook 03b versão 20261024_001223
Teste (sem sobreposição com o treino): 7,219,189 linhas
Limiares gravados na validação: [(0.01, 0.04), (0.001, 0.23)]


## 2. Previsão em lotes (com checkpoint)

A view é lida por **faixas de `id_linha`** (o id é um hash, então as faixas têm tamanho parecido; a tabela é ordenada por ele, então cada faixa só lê o que precisa). Cada lote tem ~`LOTE_LINHAS` linhas (padrão **50.000**, ~40 MiB de memória) e é previsto e salvo em `teste_lotes/`. Antes de ler cada lote, é impressa uma linha com a **memória do processo**: se o kernel morrer, é a última linha que aparece. Ao mudar `LOTE_LINHAS`, o número de lotes muda e os arquivos de uma tentativa anterior com outro número não são reaproveitados (pode apagar a pasta `teste_lotes`). O tempo de previsão de cada lote é medido (vazão em dados reais, com os núcleos configurados no modelo). Ao final, a soma das linhas dos lotes tem de ser igual ao total da view.

In [9]:
# ---------- 2.1 Previsão em lotes ----------
B = max(1, math.ceil(N_TESTE / LOTE_LINHAS)); PASTA_LOTES = PASTA_RUN / "teste_lotes"; PASTA_LOTES.mkdir(exist_ok=True)
cols_sql = ", ".join(q(c) for c in CM); ys, ps, t_pred, n_lote = [], [], [], []; X_AMOSTRA = None; reaproveitados = 0
t_inicio = time.perf_counter()
for j in range(B):
    arq = PASTA_LOTES / f"lote_{j:03d}_de_{B:03d}.npz"
    if arq.exists():
        d = np.load(arq); ys.append(d["y"]); ps.append(d["p"]); t_pred.append(float(d["t"])); n_lote.append(len(d["y"])); reaproveitados += 1
        if X_AMOSTRA is None and len(d["y"]) > 0: pass
        continue
    lo, hi = j * 2 ** 64 // B, (j + 1) * 2 ** 64 // B
    onde = f"id_linha >= {lo}" + (f" AND id_linha < {hi}" if j < B - 1 else "")
    print(f"  lote {j + 1}/{B}: lendo... (memória do processo: {rss_mib():,.0f} MiB)", flush=True)        # se o kernel morrer, a última linha mostra em que lote foi
    df = ler_csv_com_espera(f"SELECT y, {cols_sql} FROM {VIEW_TESTE} WHERE {onde}", ["y"] + CM)
    if len(df) == 0:                                            # faixa sem linhas: registra o lote vazio e segue
        np.savez(arq, y=np.zeros(0, np.int8), p=np.zeros(0, np.float32), t=0.0); ys.append(np.zeros(0, np.int8)); ps.append(np.zeros(0, np.float32)); t_pred.append(0.0); n_lote.append(0)
        print(f"  lote {j + 1}/{B}: faixa sem linhas"); continue
    X, y = df[CM].to_numpy(float), df["y"].to_numpy(np.int8); assert np.isfinite(X).all(), "valores não finitos no teste"
    t0 = time.perf_counter(); p = modelo.predict_proba(X)[:, 1].astype(np.float32); dt = time.perf_counter() - t0
    np.savez(arq, y=y, p=p, t=dt)
    ys.append(y); ps.append(p); t_pred.append(dt); n_lote.append(len(y))
    if X_AMOSTRA is None: X_AMOSTRA = X[:50_000].copy()
    print(f"  lote {j + 1}/{B}: {len(y):,} linhas lidas e previstas ({dt:.1f} s de previsão, {len(y) / dt:,.0f} fluxos/s)", flush=True)
    del df, X; gc.collect()
if reaproveitados: print(f"  {reaproveitados} lote(s) já estavam prontos e foram reaproveitados")
y_te, p_te = np.concatenate(ys), np.concatenate(ps)                 # int8 e float32: 5 vezes menos memória que int64 e float64
del ys, ps
assert len(y_te) == N_TESTE, f"as linhas dos lotes ({len(y_te):,}) não fecham com o total da view ({N_TESTE:,})"
print(f"\n✔ {len(y_te):,} previsões ({int((y_te == 0).sum()):,} BENIGN = {100 * (y_te == 0).mean():.3f}%) em {time.perf_counter() - t_inicio:.0f} s")

  lote 144/145: lendo... (memória do processo: 218 MiB)
  lote 144/145: 49,690 linhas lidas e previstas (0.2 s de previsão, 216,240 fluxos/s)
  lote 145/145: lendo... (memória do processo: 236 MiB)
  lote 145/145: 49,763 linhas lidas e previstas (0.3 s de previsão, 185,216 fluxos/s)
  143 lote(s) já estavam prontos e foram reaproveitados

✔ 7,219,189 previsões (56,216 BENIGN = 0.779%) em 9 s


## 3. Métricas no teste

Tudo é calculado **no teste como ele é**, sem balancear e sem pesos. A tabela mostra o limiar 0,5 e os limiares da validação.

In [10]:
# ---------- 3.1 Métricas ----------
PONTOS_VAL = lim_val["pontos_de_operacao_validacao"]
ordem = ["roc_auc", "recall", "taxa_falso_alarme", "especificidade", "acuracia_balanceada", "acuracia", "precisao", "f1", "mcc", "pr_auc", "precisao_benign", "f1_benign", "f1_macro", "pr_auc_benign"]
VALS, N_POS, N_NEG = contagens_por_escore(y_te, p_te)
print(f"Valores distintos de escore: {len(VALS)} | ataques: {int(N_POS.sum()):,} | BENIGN: {int(N_NEG.sum()):,}")
conjuntos = {f"limiar {LIMIAR}": metricas_hist(VALS, N_POS, N_NEG, LIMIAR)}
for pv in PONTOS_VAL: conjuntos[f"limiar da validação p/ falso alarme {pv['alvo_fpr']:.1%} ({pv['limiar']:.3f})"] = metricas_hist(VALS, N_POS, N_NEG, pv["limiar"])
tab_te = pd.DataFrame({nome: {k: m[k] for k in ordem} for nome, m in conjuntos.items()})
print("== Métricas no teste =="); display(tab_te.round(5))
cm_ = {nome: {k: int(m[k]) for k in ("tp", "fp", "tn", "fn")} for nome, m in conjuntos.items()}
print("\nMatrizes de confusão (VP = ataque detectado; FP = BENIGN chamado de ataque; VN = BENIGN correto; FN = ataque perdido):"); display(pd.DataFrame(cm_))
prev_benign = float(N_NEG.sum() / (N_POS.sum() + N_NEG.sum()))
print(f"\nLinha de base do PR-AUC do BENIGN = proporção de BENIGN no teste = {prev_benign:.5f} | o modelo obteve {conjuntos[f'limiar {LIMIAR}']['pr_auc_benign']:.5f}")
info = []
for alvo in (0.01, 0.001):                                      # APENAS informativo: limiar escolhido no próprio teste
    op = recall_em_fpr_hist(VALS, N_POS, N_NEG, alvo); info.append({"alvo_fpr": alvo, "limiar_no_teste": op["limiar"], "recall": op["recall"], "fpr": op["fpr"]})
print("\nPonto lido na curva ROC do TESTE (limiar escolhido no próprio teste; só informativo):"); display(pd.DataFrame(info).round(5))

Valores distintos de escore: 115 | ataques: 7,162,973 | BENIGN: 56,216
== Métricas no teste ==


,limiar 0.5,limiar da validação p/ falso alarme 1.0% (0.040),limiar da validação p/ falso alarme 0.1% (0.230)
roc_auc,0.99447,0.99447,0.99447
recall,0.95344,1.00000,0.99996
taxa_falso_alarme,0.01528,0.20873,0.06270
especificidade,0.98472,0.79127,0.93730
acuracia_balanceada,0.96908,0.89563,0.96863
acuracia,0.95369,0.99837,0.99947
precisao,0.99987,0.99836,0.99951
f1,0.97611,0.99918,0.99973
mcc,0.36530,0.88868,0.96521
pr_auc,0.99994,0.99994,0.99994



Matrizes de confusão (VP = ataque detectado; FP = BENIGN chamado de ataque; VN = BENIGN correto; FN = ataque perdido):


,limiar 0.5,limiar da validação p/ falso alarme 1.0% (0.040),limiar da validação p/ falso alarme 0.1% (0.230)
tp,6829486,7162961,7162681
fp,859,11734,3525
tn,55357,44482,52691
fn,333487,12,292



Linha de base do PR-AUC do BENIGN = proporção de BENIGN no teste = 0.00779 | o modelo obteve 0.98253

Ponto lido na curva ROC do TESTE (limiar escolhido no próprio teste; só informativo):


,alvo_fpr,limiar_no_teste,recall,fpr
0,0.010,0.63,0.69435,0.00977
1,0.001,0.92,0.31787,0.00094


## 4. Custo computacional na inferência
- **Vazão em dados reais:** todas as linhas do teste ÷ o tempo total de previsão dos lotes (com os `n_jobs` do modelo, que são os do treino).
- **Latência (1 fluxo por vez) e vazão em lote** com e sem pré-processamento, como no notebook 05b, numa amostra de fluxos do teste.

In [11]:
# ---------- 4.1 Custo de inferência ----------
t_total = float(sum(t_pred)); vazao_real = len(y_te) / t_total if t_total > 0 else float("nan")
print(f"Vazão em dados reais: {len(y_te):,} fluxos em {t_total:.1f} s de previsão = {vazao_real:,.0f} fluxos/s (n_jobs={modelo.n_jobs})")
if X_AMOSTRA is None:                                           # lotes já prontos de uma execução anterior: lê uma amostra só para a latência
    X_AMOSTRA = ler_csv_com_espera(f"SELECT {cols_sql} FROM {VIEW_TESTE} LIMIT 50000", CM).to_numpy(float)
X_raw = X_AMOSTRA[:, :len(prep.features)].copy()
for j, c in enumerate(prep.indicadores): X_raw[X_AMOSTRA[:, len(prep.features) + j] == 1, prep.features.index(c)] = np.nan
assert np.allclose(prep.transformar(X_raw), X_AMOSTRA), "o PreprocessadorRF não reproduz a matriz do teste"
inf_te = medir_inferencia(modelo, prep, X_AMOSTRA, X_raw, n_lat=N_LATENCIA, semente=SEMENTE)
display(pd.DataFrame(inf_te).set_index("n_jobs").T.round(3))

Vazão em dados reais: 7,219,189 fluxos em 22.3 s de previsão = 323,512 fluxos/s (n_jobs=-1)


n_jobs,1,-1
n_latencia,1000.000,1000.000
latencia_modelo_p50_ms,13.300,51.749
latencia_modelo_p95_ms,30.909,70.285
latencia_modelo_p99_ms,49.317,92.695
latencia_fim_a_fim_p50_ms,13.469,51.868
latencia_fim_a_fim_p95_ms,31.483,70.445
latencia_fim_a_fim_p99_ms,49.901,92.995
vazao_modelo_fluxos_s,136524.341,297183.307
vazao_fim_a_fim_fluxos_s,130421.659,217338.818
n_lote,49690.000,49690.000


## 5. Gravação e conferências

Grava as métricas, as previsões (`predicoes_teste.npz`, para análises futuras sem reler o banco), os gráficos (se o `matplotlib` estiver instalado) e uma linha em `registro_testes.csv`. O arquivo `metricas_teste.json` é gravado **por último**: é ele que aciona a trava de uso único.

In [12]:
# ---------- 5.1 Gravação ----------
np.savez_compressed(PASTA_RUN / "predicoes_teste.npz", y=y_te, p=p_te)
tab_te.reset_index(names="metrica").to_csv(PASTA_RUN / "metricas_teste.csv", index=False)
salvar_json({"inferencia_teste": inf_te, "vazao_dados_reais_fluxos_s": vazao_real, "tempo_previsao_total_s": t_total, "n_jobs_modelo": modelo.n_jobs}, PASTA_RUN / "custo_inferencia_teste.json")
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    d_ = slice(None, None, -1)                                   # curvas exatas a partir das contagens por escore (sem ordenar 7 milhões de linhas)
    fpr_ = np.concatenate(([0.0], np.cumsum(N_NEG[d_]) / N_NEG.sum())); tpr_ = np.concatenate(([0.0], np.cumsum(N_POS[d_]) / N_POS.sum()))
    tp_b = np.cumsum(N_NEG); rc_ = tp_b / N_NEG.sum(); pr_ = tp_b / (tp_b + np.cumsum(N_POS))
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    ax[0].plot(fpr_, tpr_); ax[0].set_xscale("symlog", linthresh=1e-4); ax[0].set_xlabel("taxa de falso alarme (BENIGN chamado de ataque)"); ax[0].set_ylabel("recall dos ataques"); ax[0].set_title("Curva ROC (teste)"); ax[0].grid(alpha=.3)
    ax[1].plot(rc_, pr_); ax[1].axhline(prev_benign, ls="--", c="gray", label=f"linha de base ({prev_benign:.4f})"); ax[1].set_xlabel("recall do BENIGN"); ax[1].set_ylabel("precisão do BENIGN"); ax[1].set_title("Precisão × recall do BENIGN (teste)"); ax[1].legend(); ax[1].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(PASTA_RUN / "curvas_teste.png", dpi=130); plt.close(fig); print("✔ gráfico salvo: curvas_teste.png")
except ImportError:
    print("(matplotlib não está instalado: gráficos não gerados)")
m0 = conjuntos[f"limiar {LIMIAR}"]
registro = {"run_id": config["run_id"], "data_teste": datetime.now().isoformat(timespec="seconds"), "n_teste": len(y_te), "benign_teste": int((y_te == 0).sum()), "limiar": LIMIAR,
            **{k: m0[k] for k in ("roc_auc", "recall", "taxa_falso_alarme", "especificidade", "acuracia_balanceada", "acuracia", "precisao", "f1", "mcc", "pr_auc", "precisao_benign", "f1_benign", "f1_macro", "pr_auc_benign")},
            "vazao_dados_reais_fluxos_s": vazao_real, "pasta": str(PASTA_RUN)}
reg_csv = PASTA_BASE / "registro_testes.csv"; pd.DataFrame([registro]).to_csv(reg_csv, mode="a", header=not reg_csv.exists(), index=False)
salvar_json({"run_id": config["run_id"], "n_teste": len(y_te), "limiar_padrao": LIMIAR, "metricas": conjuntos, "linha_de_base_pr_auc_benign": prev_benign, "pontos_informativos_no_teste": info,
             "limiares_da_validacao": PONTOS_VAL, "observacao": "teste sem sobreposição com o treino, sem balancear e sem pesos"}, RESULTADO)

ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
verifica("Previsões = linhas do teste", len(y_te) == N_TESTE, f"({len(y_te):,})")
verifica("Probabilidades entre 0 e 1", bool(((p_te >= 0) & (p_te <= 1)).all()))
verifica("Matrizes de confusão fecham com o total", all(sum(v.values()) == N_TESTE for v in cm_.values()))
verifica("Lotes gravados somam o teste", sum(n_lote) == N_TESTE and len(list(PASTA_LOTES.glob(f"lote_*_de_{B:03d}.npz"))) == B, f"({B} lotes)")
verifica("Modelo final avaliado é o gravado pelo notebook 05b", config["run_id"] == PASTA_RUN.name.replace("rf_bin_", ""))
verifica("Arquivos gravados", all((PASTA_RUN / f).exists() for f in ("predicoes_teste.npz", "metricas_teste.csv", "custo_inferencia_teste.json", "metricas_teste.json")))
_rel = np.load(PASTA_RUN / "predicoes_teste.npz"); _v, _a, _b = contagens_por_escore(_rel["y"], _rel["p"])
verifica("Relido: predicoes_teste.npz reproduz as métricas", abs(metricas_hist(_v, _a, _b, LIMIAR)["mcc"] - m0["mcc"]) < 1e-9)
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas. Resultados em {PASTA_RUN}")

✔ gráfico salvo: curvas_teste.png
✔ Previsões = linhas do teste (7,219,189)
✔ Probabilidades entre 0 e 1 
✔ Matrizes de confusão fecham com o total 
✔ Lotes gravados somam o teste (145 lotes)
✔ Modelo final avaliado é o gravado pelo notebook 05b 
✔ Arquivos gravados 
✔ Relido: predicoes_teste.npz reproduz as métricas 

✔ 7/7 checagens aprovadas. Resultados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_binario_com_auc\rf_bin_20261024_002027
